# K-Means Clustering - Allrecipes.com

### Question:
How effectively can K-Means clustering be applied to segment recipes, based on macronutrient composition and preparation time, in order to support time-constrained individuals in making nutritionally informed choices?

#### Overview:
To answer the question above, the [all_recipes.csv](https://github.com/owlzyseyes/tastyR/tree/main/data-raw) data, which was originally scraped from [www.allrecipes.com](https://www.allrecipes.com/), and featured on [https://www.brians.works/i-scraped-14k-recipes-so-you-wont-have-to/](https://www.brians.works/i-scraped-14k-recipes-so-you-wont-have-to/), was provided to [Github](https://github.com/) by [owlzeyes - Brian Mubia](https://github.com/owlzyseyes) and for this project was taken from the [tastyR package](https://github.com/owlzyseyes/tastyR) (see also [tastyR package](https://cran.r-project.org/package=tastyR)).

The original [all_recipes.csv](https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/allrecipes.csv) file contains 14,426 rows and 16 columns of data. This data was chosen, rather than the smaller more curated [cuisines.csv](https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/cuisines.csv) file due to the requirement for demonstrating data transformation and cleansing skills. The smaller file contains 2,218 rows and 17 columns of data with the additional column being a character column 'country - The country/region the cuisine is from' and won't be used in this analysis.

*Write an overview/summary of the steps taken in here when it's finished.*

## 1. Libraries and Data Loading

In [210]:
import warnings
warnings.filterwarnings("ignore") # suppress non-critical warnings throughout the script.

# Import today's date to use in the work.
from datetime import date

In [211]:
# Standard library imports
import importlib
import subprocess
import sys

# List required packages
required_packages = [
    "pandas",          # DataFrames, joins, light transforms
    "numpy",           # Numerical ops
    "matplotlib",      # Plotting
    "seaborn",         # Statistical plots
    "sklearn",         # ML
    "statsmodels",     # Statistical modelling
    "ydata_profiling", # EDA (optional)
    "ydata-sdk",       # Advanced data profiling, validation, and synthetic data
]

"""
The ensure_packages function below is set up to check whether each package in the above list is installed.
- If installed it prints a confirmation message with a tick.
- If missing it installs the package using pip install. 
"""
def ensure_packages(packages):
    for pkg in packages:
        try:
            importlib.import_module(pkg)
            print(f"✓ {pkg} already installed")
        except ImportError:
            print(f"✗ {pkg} missing — installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Run the package check.
ensure_packages(required_packages)

✓ pandas already installed
✓ numpy already installed
✓ matplotlib already installed
✓ seaborn already installed
✓ sklearn already installed
✓ statsmodels already installed
✓ ydata_profiling already installed
✗ ydata-sdk missing — installing...


In [212]:
# Import the libraries required for this work
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [213]:
df = pd.read_csv('https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/allrecipes.csv')
df.head(10)

,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,Chewy Whole Wheat Peanut Butter Brownies,https://www.allrecipes.com/recipe/140717/chewy...,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ...",222.0,13.0,24.0,6.0,4.4,47.0,36.0,20,35,55,16.0
1,Pumpkin Pie Eggnog,https://www.allrecipes.com/recipe/269204/pumpk...,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ ...",477.0,31.0,43.0,8.0,5.0,1.0,1.0,10,5,495,8.0
2,Eggs Poached in Tomato Sauce,https://www.allrecipes.com/recipe/238054/eggs-...,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion...",354.0,18.0,32.0,20.0,4.8,4.0,4.0,10,75,85,4.0
3,Minestrone Casserole,https://www.allrecipes.com/minestrone-casserol...,Sarah Brekke,2025-03-03,4 cups dried mafalda pasta (mini lasagna noodl...,356.0,9.0,53.0,19.0,4.3,14.0,13.0,20,40,60,8.0
4,Yummy Stuffed Peppers,https://www.allrecipes.com/recipe/241937/yummy...,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and se...",366.0,22.0,23.0,19.0,4.7,84.0,67.0,30,95,125,8.0
5,Prime Rib Our Way,https://www.allrecipes.com/recipe/219586/prime...,Cajun Girl,2019-04-03,"1 (8 pound) standing rib roast, fat trimmed, 2...",709.0,47.0,31.0,37.0,4.2,5.0,3.0,45,80,155,12.0
6,Parmesan Chicken Legs,https://www.allrecipes.com/recipe/8636/parmesa...,Anika,2023-01-04,"1 large egg, 2 cups grated Parmesan cheese, 1 ...",466.0,27.0,1.0,52.0,4.4,648.0,468.0,15,45,60,6.0
7,Chicken Andouille Gumbo,https://www.allrecipes.com/recipe/12950/chicke...,Bob Cody,2022-07-14,"12 cups water, 3 pounds chicken parts, 2 table...",782.0,61.0,19.0,40.0,4.6,347.0,259.0,10,190,200,8.0
8,Sweet Pork for Burritos,https://www.allrecipes.com/recipe/185816/sweet...,Dean,2023-01-19,"3 pounds pork shoulder roast, 2 cups salsa, 1 ...",355.0,15.0,33.0,23.0,4.7,129.0,102.0,30,480,510,12.0
9,Quick Baked Chicken Parmesan,https://www.allrecipes.com/recipe/239507/quick...,ChristineM,2024-11-14,"canola oil cooking spray, ½ cup water, 1 egg,...",395.0,12.0,33.0,37.0,4.7,195.0,153.0,20,55,75,6.0


In [214]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14426 entries, 0 to 14425
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            14426 non-null  object 
 1   url             14426 non-null  object 
 2   author          14426 non-null  object 
 3   date_published  14426 non-null  object 
 4   ingredients     14417 non-null  object 
 5   calories        14226 non-null  float64
 6   fat             14070 non-null  float64
 7   carbs           14212 non-null  float64
 8   protein         14178 non-null  float64
 9   avg_rating      13454 non-null  float64
 10  total_ratings   13454 non-null  float64
 11  reviews         13353 non-null  float64
 12  prep_time       14426 non-null  int64  
 13  cook_time       14426 non-null  int64  
 14  total_time      14426 non-null  int64  
 15  servings        14405 non-null  float64
dtypes: float64(8), int64(3), object(5)
memory usage: 1.8+ MB


During the process we will generate some labels/descriptions for the recipes therefore we split the data (df_full = df + df_features). There is no unique_id column in the original data, therefore we create one now for any joining/merging required later. 

In [215]:
# Create a unique_id for each row.
df = df.reset_index(drop=True)
df['unique_id'] = df.index

# Move 'unique_id' to the first position
cols = ['unique_id'] + [col for col in df.columns if col != 'unique_id']
df = df[cols]

df.head(5)

,unique_id,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,0,Chewy Whole Wheat Peanut Butter Brownies,https://www.allrecipes.com/recipe/140717/chewy...,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ...",222.0,13.0,24.0,6.0,4.4,47.0,36.0,20,35,55,16.0
1,1,Pumpkin Pie Eggnog,https://www.allrecipes.com/recipe/269204/pumpk...,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ ...",477.0,31.0,43.0,8.0,5.0,1.0,1.0,10,5,495,8.0
2,2,Eggs Poached in Tomato Sauce,https://www.allrecipes.com/recipe/238054/eggs-...,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion...",354.0,18.0,32.0,20.0,4.8,4.0,4.0,10,75,85,4.0
3,3,Minestrone Casserole,https://www.allrecipes.com/minestrone-casserol...,Sarah Brekke,2025-03-03,4 cups dried mafalda pasta (mini lasagna noodl...,356.0,9.0,53.0,19.0,4.3,14.0,13.0,20,40,60,8.0
4,4,Yummy Stuffed Peppers,https://www.allrecipes.com/recipe/241937/yummy...,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and se...",366.0,22.0,23.0,19.0,4.7,84.0,67.0,30,95,125,8.0


K-Means clustering models only work with numeric, scaled data therefore the next step is to extract numeric nutritional columns (calories, fat, carbs, protein) and the time columns (prep_time, cook_time and total_time).

Three time based columns are being retained for now until further investigation can be conducted into the total_time column; if that is completed well, then the others will not be needed but if some of the values appear to be incorrect, then the other two columns may be useful for calculating the total_time and imputing the value.

- **df_full** (containing all columns).
- **df** (the columns required for the model building).
- **df_features** (the remaining columns).

In [216]:
df_full = df.copy()
df_full.head(5)

,unique_id,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,0,Chewy Whole Wheat Peanut Butter Brownies,https://www.allrecipes.com/recipe/140717/chewy...,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ...",222.0,13.0,24.0,6.0,4.4,47.0,36.0,20,35,55,16.0
1,1,Pumpkin Pie Eggnog,https://www.allrecipes.com/recipe/269204/pumpk...,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ ...",477.0,31.0,43.0,8.0,5.0,1.0,1.0,10,5,495,8.0
2,2,Eggs Poached in Tomato Sauce,https://www.allrecipes.com/recipe/238054/eggs-...,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion...",354.0,18.0,32.0,20.0,4.8,4.0,4.0,10,75,85,4.0
3,3,Minestrone Casserole,https://www.allrecipes.com/minestrone-casserol...,Sarah Brekke,2025-03-03,4 cups dried mafalda pasta (mini lasagna noodl...,356.0,9.0,53.0,19.0,4.3,14.0,13.0,20,40,60,8.0
4,4,Yummy Stuffed Peppers,https://www.allrecipes.com/recipe/241937/yummy...,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and se...",366.0,22.0,23.0,19.0,4.7,84.0,67.0,30,95,125,8.0


In [217]:
cols_to_keep = [
    'unique_id', 'calories', 'fat', 'carbs', 'protein',
    'prep_time', 'cook_time', 'total_time'
]

df_core = df_full[cols_to_keep]

# Everything except the core columns
cols_features = [c for c in df_full.columns if c not in cols_to_keep]
df_features = df_full[cols_features]

In [218]:
df_core.head(5)

,unique_id,calories,fat,carbs,protein,prep_time,cook_time,total_time
0,0,222.0,13.0,24.0,6.0,20,35,55
1,1,477.0,31.0,43.0,8.0,10,5,495
2,2,354.0,18.0,32.0,20.0,10,75,85
3,3,356.0,9.0,53.0,19.0,20,40,60
4,4,366.0,22.0,23.0,19.0,30,95,125


In [219]:
df_features.head(5)

,name,url,author,date_published,ingredients,avg_rating,total_ratings,reviews,servings
0,Chewy Whole Wheat Peanut Butter Brownies,https://www.allrecipes.com/recipe/140717/chewy...,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ...",4.4,47.0,36.0,16.0
1,Pumpkin Pie Eggnog,https://www.allrecipes.com/recipe/269204/pumpk...,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ ...",5.0,1.0,1.0,8.0
2,Eggs Poached in Tomato Sauce,https://www.allrecipes.com/recipe/238054/eggs-...,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion...",4.8,4.0,4.0,4.0
3,Minestrone Casserole,https://www.allrecipes.com/minestrone-casserol...,Sarah Brekke,2025-03-03,4 cups dried mafalda pasta (mini lasagna noodl...,4.3,14.0,13.0,8.0
4,Yummy Stuffed Peppers,https://www.allrecipes.com/recipe/241937/yummy...,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and se...",4.7,84.0,67.0,8.0


## EDA - df_core

Look for missing values across the nutrtion and time variables, try to fill missing values in where possible. Check for outliers.

In [220]:
# Identify columns with null values
columns_with_nulls = df_core.columns[df_core.isnull().any()].tolist()

# Print column names containing null values
print("Columns with null values:", columns_with_nulls)

Columns with null values: ['calories', 'fat', 'carbs', 'protein']


Nine columns contain NaN values, listed above, let's look at them.

In [221]:
df_core.isnull().mean()

unique_id     0.000000
calories      0.013864
fat           0.024678
carbs         0.014834
protein       0.017191
prep_time     0.000000
cook_time     0.000000
total_time    0.000000
dtype: float64

Check how many of the rows with null values are null for all of them.

In [222]:
filtered_df = df_core[df_core[columns_with_nulls].isna().all(axis=1)]
filtered_df

,unique_id,calories,fat,carbs,protein,prep_time,cook_time,total_time
59,59,NaN,NaN,NaN,NaN,5,0,5
110,110,NaN,NaN,NaN,NaN,20,55,315
127,127,NaN,NaN,NaN,NaN,10,0,25
211,211,NaN,NaN,NaN,NaN,25,30,55
363,363,NaN,NaN,NaN,NaN,5,0,485
...,...,...,...,...,...,...,...,...
14060,14060,NaN,NaN,NaN,NaN,10,20,30
14105,14105,NaN,NaN,NaN,NaN,10,0,10
14185,14185,NaN,NaN,NaN,NaN,10,0,10
14317,14317,NaN,NaN,NaN,NaN,20,15,35


These will be of no use to our model as they contain too many mussing values in the nutrition columns.
These must be removed from our original data.

In [223]:
# Example: remove rows where fat, protein, and carbs are all NaN
df_core = df_core[~df_core[columns_with_nulls].isna().all(axis=1)]
df_core

,unique_id,calories,fat,carbs,protein,prep_time,cook_time,total_time
0,0,222.0,13.0,24.0,6.0,20,35,55
1,1,477.0,31.0,43.0,8.0,10,5,495
2,2,354.0,18.0,32.0,20.0,10,75,85
3,3,356.0,9.0,53.0,19.0,20,40,60
4,4,366.0,22.0,23.0,19.0,30,95,125
...,...,...,...,...,...,...,...,...
14421,14421,338.0,25.0,15.0,14.0,10,45,55
14422,14422,572.0,37.0,28.0,30.0,30,40,70
14423,14423,244.0,13.0,22.0,11.0,10,25,35
14424,14424,401.0,20.0,45.0,5.0,40,90,1585


Note **200 rows** have been removed.

Next look at time, recipes cannot be meaningfully used where there is no time component in any of the time columns. This shall be treated as too much missing information to be useful as in the very least there should be some preparation time to create a meal.

In [224]:
# Filter rows where total_time == 0
df_zero_time = df_core[(df_core['total_time'] == 0) & (df_core['prep_time'] == 0) & (df_core['cook_time'] == 0)]
df_zero_time

,unique_id,calories,fat,carbs,protein,prep_time,cook_time,total_time
18,18,347.0,20.0,40.0,6.0,0,0,0
58,58,401.0,19.0,56.0,4.0,0,0,0
66,66,332.0,16.0,41.0,9.0,0,0,0
73,73,171.0,10.0,21.0,2.0,0,0,0
85,85,346.0,19.0,32.0,13.0,0,0,0
...,...,...,...,...,...,...,...,...
14262,14262,517.0,11.0,67.0,34.0,0,0,0
14266,14266,211.0,9.0,31.0,2.0,0,0,0
14341,14341,149.0,9.0,18.0,1.0,0,0,0
14397,14397,223.0,6.0,41.0,2.0,0,0,0


345 rows will be removed from the dataset.

In [225]:
# Filter rows where total_time == 0
df_core = df_core[(df_core['total_time'] > 0)]
df_core

,unique_id,calories,fat,carbs,protein,prep_time,cook_time,total_time
0,0,222.0,13.0,24.0,6.0,20,35,55
1,1,477.0,31.0,43.0,8.0,10,5,495
2,2,354.0,18.0,32.0,20.0,10,75,85
3,3,356.0,9.0,53.0,19.0,20,40,60
4,4,366.0,22.0,23.0,19.0,30,95,125
...,...,...,...,...,...,...,...,...
14421,14421,338.0,25.0,15.0,14.0,10,45,55
14422,14422,572.0,37.0,28.0,30.0,30,40,70
14423,14423,244.0,13.0,22.0,11.0,10,25,35
14424,14424,401.0,20.0,45.0,5.0,40,90,1585


In [231]:
# Filter rows where total_time == 0
df_core = df_core[(df_core['total_time'] != (df_core['prep_time']+df_core['cook_time']))].sort_values(by='total_time', ascending=False)
df_core.head(50)

,unique_id,calories,fat,carbs,protein,prep_time,cook_time,total_time
11796,11796,39.0,0.0,9.0,2.0,60,90,31160
11479,11479,58.0,0.0,14.0,1.0,45,15,30300
928,928,1077.0,46.0,131.0,11.0,45,240,24780
14028,14028,728.0,27.0,110.0,9.0,20,150,20810
7911,7911,8.0,NaN,2.0,1.0,15,20,20675
7118,7118,429.0,19.0,59.0,6.0,15,90,20265
13500,13500,422.0,14.0,67.0,5.0,70,90,17950
3526,3526,70.0,0.0,17.0,2.0,15,0,14535
10840,10840,34.0,0.0,8.0,1.0,30,0,14440
12115,12115,53.0,0.0,12.0,2.0,30,0,12990


Many of the recipes have a total_time does not add to the sum of prep_time and cook_time. In these instances where the total_time is more than 10% over the prep time plus the cook time, we will replace the total time with prep time plus cook time.

In [234]:
# Calculate the sum of prep_time and cook_time
sum_times = df_core['prep_time'] + df_core['cook_time']

# Apply the condition: if total_time > 1.1 * (prep_time + cook_time), replace it
df_core['total_time'] = df_core.apply(
    lambda row: sum_times[row.name] if row['total_time'] > 1.1 * sum_times[row.name] else row['total_time'],
    axis=1
)

df_core = df_core.sort_values(by='total_time', ascending=False)
df_core.head(20)

,unique_id,calories,fat,carbs,protein,prep_time,cook_time,total_time
6904,6904,194.0,5.0,16.0,22.0,10,3000,3025.0
12376,12376,182.0,14.0,0.0,14.0,10,1460,1480.0
12186,12186,204.0,5.0,20.0,20.0,30,1390,1420.0
5039,5039,518.0,24.0,1.0,69.0,10,1110,1130.0
6660,6660,527.0,37.0,1.0,45.0,20,1090,1120.0
10224,10224,2830.0,70.0,31.0,519.0,45,960,1020.0
11281,11281,157.0,4.0,27.0,4.0,5,840,905.0
4742,4742,390.0,16.0,51.0,13.0,55,720,790.0
13040,13040,1129.0,51.0,118.0,53.0,30,720,780.0
2070,2070,411.0,44.0,3.0,2.0,5,720,725.0


In [206]:
# Sort by time ascending
df_sorted = df_core.sort_values(by='total_time', ascending=False).reset_index(drop=True)
df_sorted

,unique_id,calories,fat,carbs,protein,prep_time,cook_time,total_time
0,11796,39.0,0.0,9.0,2.0,60,90,31160
1,11479,58.0,0.0,14.0,1.0,45,15,30300
2,928,1077.0,46.0,131.0,11.0,45,240,24780
3,14028,728.0,27.0,110.0,9.0,20,150,20810
4,7911,8.0,NaN,2.0,1.0,15,20,20675
...,...,...,...,...,...,...,...,...
13152,7138,521.0,40.0,39.0,5.0,10,1,11
13153,10171,273.0,9.0,7.0,38.0,5,6,11
13154,2923,177.0,11.0,14.0,7.0,10,1,11
13155,8561,71.0,3.0,7.0,3.0,10,1,11


In [50]:
df.describe(include='object')

,name,url,author,date_published,ingredients
count,14426,14426,14426,14426,14417
unique,14401,14426,8297,1542,14411
top,Deviled Egg Potato Salad,https://www.allrecipes.com/recipe/236988/chef-...,John Mitzewich,2022-07-14,"¾ cup brown sugar, ⅓ cup all-purpose flour, 1 ..."
freq,2,1,662,645,2


In [51]:
df.describe(include=['float64', 'int64'])

,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
count,14226.000000,14070.000000,14212.000000,14178.000000,13454.000000,13454.000000,13353.000000,14426.000000,14426.000000,14426.000000,14405.000000
mean,344.878532,17.835537,32.863425,14.422344,4.525606,102.620262,94.418483,17.345002,42.515943,144.065645,11.028393
std,250.020621,16.679835,27.590609,17.528592,0.409127,172.979317,164.522217,24.872135,96.819185,874.250121,13.026444
min,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,181.000000,7.000000,14.000000,3.000000,4.400000,5.000000,5.000000,10.000000,10.000000,30.000000,4.000000
50%,307.000000,14.000000,29.000000,8.000000,4.600000,26.000000,24.000000,15.000000,20.000000,55.000000,8.000000
75%,453.000000,24.000000,45.000000,22.000000,4.800000,112.000000,100.000000,20.000000,45.000000,100.000000,12.000000
max,9538.000000,612.000000,746.000000,939.000000,5.000000,997.000000,999.000000,2160.000000,4325.000000,60485.000000,300.000000


Next Steps:
1. Explore the data, deciding which columns are required.
2. Cleanse the required columns dealing with (missing values etc etc)
3. Move to k-means clustering.
4. Write report.